## Этап 0. Импорты и конфигурация

Здесь подключаются все сервисы проекта, константы путей и параметры запуска пайплайна.


In [1]:
from logging import getLogger
from pathlib import Path

from src.config import BASE_DIR, cfg
from src.domain.constans import DataConstants
from src.domain.training import TrainingConfig
from src.services.data_utils import TextDataPreparation
from src.services.lstm_trainer import LSTMTrainerService
from src.services.report_service import FinalReportService
from src.services.rouge_service import get_metric_service
from src.services.tokens import get_tokens_service
from src.services.transformer_baseline import DistilGPT2BaselineService
from src.services.training_dataset import (
    DataLoaderFactory,
    NextTokenDataset,
)

logger = getLogger(__name__)

raw_data_file = Path(BASE_DIR / DataConstants.raw_data_patch)
dataset_processed_file = Path(BASE_DIR / DataConstants.dataset_processed_patch)
xy_dataset_file = Path(BASE_DIR / DataConstants.X_Y_dataset_file)
train_output_path = Path(BASE_DIR / DataConstants.train_output_path)
val_output_path = Path(BASE_DIR / DataConstants.val_output_path)
test_output_path = Path(BASE_DIR / DataConstants.test_output_path)
model_output_path = Path(BASE_DIR / "models" / "next_token_lstm.pt")
report_output_path = Path(BASE_DIR / "reports" / "final_report.txt")


/home/ubuntu/server/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Этап 1. Подготовка данных

Функция `prepare_datasets` выполняет:
- очистку исходного датасета,
- построение next-token датасета `X -> Y`,
- разделение на `train / val / test`.


In [2]:
def prepare_datasets(recreate_data: bool = False) -> dict[str, str]:
    text_service = TextDataPreparation(recreate_data=recreate_data)

    result = text_service.create_process_dataset(
        input_path=raw_data_file,
        output_path=dataset_processed_file,
        batch_size=DataConstants.batch_size,
        min_text_length=DataConstants.X_length + 1,
    )
    process_dataset_result = result
    logger.info(result)

    result = text_service.create_next_token_dataset(
        input_path=dataset_processed_file,
        output_path=xy_dataset_file,
        batch_size=DataConstants.batch_size,
    )
    xy_dataset_result = result
    logger.info(result)

    result = text_service.split_train_val_test(
        input_path=xy_dataset_file,
        train_output_path=train_output_path,
        val_output_path=val_output_path,
        test_output_path=test_output_path,
    )
    split_result = result
    logger.info(result)
    return {
        "process_dataset_result": process_dataset_result,
        "xy_dataset_result": xy_dataset_result,
        "split_result": split_result,
    }


def get_config_snapshot() -> dict[str, str]:
    return {
        "LOCAL_DEVELOP": str(cfg.local_develop),
        "RECREATE_DATA": str(cfg.recreate_data),
        "LOCAL_DEVELOP_LINE_LIMIT": str(cfg.local_develop_line_limit),
        "TRAINING_BATCH_SIZE": str(cfg.training_batch_size),
        "TRAINING_EPOCHS": str(cfg.training_epochs),
        "TRAINING_PROMPT_FRACTION": str(cfg.training_prompt_fraction),
        "TRAINING_WEIGHT_DECAY": str(cfg.training_weight_decay),
        "TRAINING_NUM_EXAMPLES": str(cfg.training_num_examples),
        "TRAINING_MAX_NEW_TOKENS": str(cfg.training_max_new_tokens),
        "TRAINING_DATASET_CHUNKSIZE": str(cfg.training_dataset_chunksize),
        "LSTM_EMBEDDING_DIM": str(cfg.lstm_embedding_dim),
        "LSTM_HIDDEN_DIM": str(cfg.lstm_hidden_dim),
        "LSTM_NUM_LAYERS": str(cfg.lstm_num_layers),
        "LSTM_DROPOUT": str(cfg.lstm_dropout),
        "LSTM_LEARNING_RATE": str(cfg.lstm_learning_rate),
        "DATA_X_LENGTH": str(DataConstants.X_length),
        "DATA_BATCH_SIZE_PREP": str(DataConstants.batch_size),
    }


## Этап 2-4. Обучение LSTM и сравнение с DistilGPT2

Функция `run_training`:
- создаёт `Dataset` и `DataLoader`,
- обучает LSTM и считает метрики,
- запускает baseline `distilgpt2` через `transformers.pipeline`,
- возвращает результаты для отчёта.


In [3]:
def run_training() -> tuple:
    tokens_service = get_tokens_service()
    metric_service = get_metric_service()

    config = TrainingConfig(
        batch_size=cfg.training_batch_size,
        epochs=cfg.training_epochs,
        prompt_fraction=cfg.training_prompt_fraction,
        weight_decay=cfg.training_weight_decay,
        num_examples=cfg.training_num_examples,
        max_new_tokens=cfg.training_max_new_tokens,
        embedding_dim=cfg.lstm_embedding_dim,
        hidden_dim=cfg.lstm_hidden_dim,
        num_layers=cfg.lstm_num_layers,
        dropout=cfg.lstm_dropout,
        learning_rate=cfg.lstm_learning_rate,
    )

    train_dataset = NextTokenDataset(
        train_output_path,
        chunksize=cfg.training_dataset_chunksize,
        shuffle_buffer_size=cfg.training_dataset_chunksize,
    )
    val_dataset = NextTokenDataset(
        val_output_path,
        chunksize=cfg.training_dataset_chunksize,
    )
    test_dataset = NextTokenDataset(
        test_output_path,
        chunksize=cfg.training_dataset_chunksize,
    )

    dataloader_factory = DataLoaderFactory()
    train_loader = dataloader_factory.create(
        dataset=train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
    )
    val_loader = dataloader_factory.create(
        dataset=val_dataset,
        batch_size=config.batch_size,
        shuffle=False,
    )
    test_loader = dataloader_factory.create(
        dataset=test_dataset,
        batch_size=config.batch_size,
        shuffle=False,
    )

    trainer = LSTMTrainerService(
        tokens_service=tokens_service,
        metric_service=metric_service,
        config=config,
        model_output_path=model_output_path,
    )

    result = trainer.run(
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        test_dataset=test_dataset,
    )

    logger.info(
        "Лучшие параметры: emb=%s hid=%s layers=%s dropout=%s lr=%s",
        result.best_params.embedding_dim,
        result.best_params.hidden_dim,
        result.best_params.num_layers,
        result.best_params.dropout,
        result.best_params.learning_rate,
    )
    logger.info(
        "Test metrics: loss=%.4f rouge1=%.4f rouge2=%.4f",
        result.test_metrics.loss,
        result.test_metrics.rouge1,
        result.test_metrics.rouge2,
    )
    logger.info("Модель сохранена: %s", result.best_model_path)

    baseline_service = DistilGPT2BaselineService(
        tokens_service=tokens_service,
        metric_service=metric_service,
    )
    baseline_val_metrics, baseline_test_metrics = baseline_service.run(
        val_dataset=val_dataset,
        test_dataset=test_dataset,
        examples_count=config.num_examples,
        max_new_tokens=config.max_new_tokens,
    )
    logger.info(
        "DistilGPT2 baseline val: rouge1=%.4f rouge2=%.4f | test: rouge1=%.4f rouge2=%.4f",
        baseline_val_metrics.rouge1,
        baseline_val_metrics.rouge2,
        baseline_test_metrics.rouge1,
        baseline_test_metrics.rouge2,
    )
    return result, baseline_val_metrics, baseline_test_metrics


## Этап 5. Формирование итогового отчёта

Функция `run_pipeline` связывает все этапы и вызывает `FinalReportService`,
который сохраняет результаты и выводы в `reports/final_report.txt`.


In [4]:
config_snapshot = get_config_snapshot()
prepare_results = prepare_datasets(recreate_data=cfg.recreate_data)
train_result, baseline_val_result, baseline_test_result = run_training()
report_service = FinalReportService(report_output_path=report_output_path)
report_service.create(
    config_snapshot,
    prepare_results,
    train_result,
    baseline_val_result,
    baseline_test_result,
)


Prepare raw csv: 1599batch [00:03, 423.28batch/s]
2026-03-21 18:03:40,716 | INFO | __main__ | Создан новый файл csv с сырыми данными, количество строк - 80000
Build X/Y dataset: 1600chunk [00:29, 54.85chunk/s]
2026-03-21 18:04:10,863 | INFO | __main__ | Создан датасет X->Y, количество строк - 3190620
2026-03-21 18:04:19,706 | INFO | __main__ | Разделение завершено: train=2552496, val=319062, test=319062, total=3190620
2026-03-21 18:04:19,707 | INFO | absl | Using default tokenizer.
2026-03-21 18:04:19,731 | INFO | src.services.lstm_trainer | Интерфейс для модели - cuda
2026-03-21 18:04:20,639 | INFO | src.services.lstm_trainer | Батчей на эпоху: train=2493 val=312 test=312 | train батчей за все эпохи=24930
LSTM epochs:   0%|          | 0/10 [00:00<?, ?epoch/s]

Epoch 01/10 | train_loss=6.3758 | val_loss=6.0551 | accuracy=0.1215 | val_rouge1=0.1226 | val_rouge2=0.0000 | test_loss=6.0623


LSTM epochs:  10%|█         | 1/10 [03:53<35:04, 233.88s/epoch]

Epoch 02/10 | train_loss=6.0076 | val_loss=5.9485 | accuracy=0.1327 | val_rouge1=0.1338 | val_rouge2=0.0000 | test_loss=5.9563


LSTM epochs:  20%|██        | 2/10 [07:44<30:55, 231.90s/epoch]

...

Device set to use cuda:0
2026-03-21 18:42:54,037 | INFO | src.services.transformer_baseline | DistilGPT2 baseline инициализирован через transformers.pipeline
Baseline val: 100%|██████████| 500/500 [00:04<00:00, 123.58sample/s]



DistilGPT2 baseline (val): rouge1=0.0880, rouge2=0.0000

Примеры DistilGPT2 baseline:
[1] input:  it is drizzling guess
    target: that
    pred:   work
[2] input:  hard time being inspired this morning a
    target: little
    pred:   lot
[3] input:  twitter chattingsurfing on 128
    target: ##k
    pred:   ,
[4] input:  throwing a party for kim chiu
    target: then
    pred:   
[5] input:  no garden spanish is crap without jonathan
    target: xx
    pred:   .
[6] input:  of his computer on the last day
    target: of
    pred:   of
[7] input:  only casual i hope i able to
    target: watch
    pred:   do
[8] input:  home from skool yesterdai
    target: ##i
    pred:   y


Baseline test: 100%|██████████| 500/500 [00:03<00:00, 129.00sample/s]
2026-03-21 18:43:02,378 | INFO | __main__ | DistilGPT2 baseline val: rouge1=0.0880 rouge2=0.0000 | test: rouge1=0.0840 rouge2=0.0000
2026-03-21 18:43:02,387 | INFO | src.services.report_service | Отчет сохранен: /home/ubuntu/server/reports/final_report_20260321_184302.txt



DistilGPT2 baseline (test): rouge1=0.0840, rouge2=0.0000

Примеры DistilGPT2 baseline:
[1] input:  give jason mraz a kiss from
    target: me
    pred:   a
[2] input:  ##t now but no spoilers for
    target: supernatural
    pred:   the
[3] input:  girl watching break in her new wii
    target: workout
    pred:   it
[4] input:  are leaving beach week already for graduation
    target: being
    pred:   
[5] input:  do you is about to go to
    target: gymnastics
    pred:   the
[6] input:  i want the pasta u ordered so
    target: hungry
    pred:   far
[7] input:  think thats really possible more like
    target: sm
    pred:   you
[8] input:  still hot as hell the magic broke
    target: my
    pred:   out


PosixPath('/home/ubuntu/server/reports/final_report_20260321_184302.txt')